# FSR v2 Chunking Strategy Diagnostic

**Purpose**: Diagnose why `v1_hierarchical→recursive (import_failed: NotebookImportException)` appears in chunk table.

This notebook:
1. Inspects chunk data and fallback strategy strings
2. Compares chunking strategies side-by-side
3. Analyzes what causes v1_hierarchical import failures
4. Validates chunk quality across strategies
5. Recommends fixes for failed conversions

**Key Finding**: The chunk_strategy value `v1_hierarchical→recursive (import_failed: NotebookImportException)` tells us:
- ✅ Fallback logging is working correctly
- ❌ But hierarchical_chunking_v1 module cannot be imported in the chunking job environment
- Solution: Fix the import path in chunker.py or ensure module availability

In [ ]:
# Section 1: Import Required Libraries and Initialize Parameters

import pandas as pd
import json
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Initialize Databricks parameters
METADATA_TABLE_V2 = "vaid.ai_sot_field_service_report.fsr_metadata_v2"

# Strategy-specific chunk tables
CHUNK_TABLES = {
    "recursive": "vaid.ai_std_con_field_service_report.fsr_chunks_v2",
    "character": "vaid.ai_std_con_field_service_report.fsr_chunks_v2_char",
    "markdown": "vaid.ai_std_con_field_service_report.fsr_chunks_v2_md",
    "section": "vaid.ai_std_con_field_service_report.fsr_chunks_v2_sec",
    "v1_hierarchical": "vaid.ai_std_con_field_service_report.fsr_chunks_v2_v1h",
}

print("✓ Initialized parameters:")
print(f"  METADATA_TABLE: {METADATA_TABLE_V2}")
print(f"  CHUNK_TABLES: {list(CHUNK_TABLES.keys())}")

## Section 2: Inspect Fallback Strategies and Import Failures

Query the chunk table to find rows where v1_hierarchical fell back to recursive.

In [ ]:
# Target document (replace with your document_id)
TARGET_DOC_ID = "35803273-2440-4d36-87fe-e45f7f0e5467_605011422-40815-270t483-final_master_report"

# Query v1_hierarchical table for fallback strategies
fallback_query = f"""
SELECT 
    chunk_id,
    chunk_index,
    document_id,
    chunk_strategy,
    page_number,
    LENGTH(chunk_text) as chunk_size,
    created_at
FROM {CHUNK_TABLES['v1_hierarchical']}
WHERE document_id = '{TARGET_DOC_ID}'
  AND chunk_strategy LIKE '%import_failed%'
LIMIT 20;
"""

print(f"Querying for fallback strategies in document: {TARGET_DOC_ID}\n")
fallback_df = spark.sql(fallback_query).toPandas()

if len(fallback_df) > 0:
    print(f"❌ Found {len(fallback_df)} chunks with import_failed fallback:\n")
    print(fallback_df[['chunk_index', 'chunk_strategy', 'chunk_size', 'page_number']])
    print("\n" + "="*80)
    print("FALLBACK REASON BREAKDOWN:")
    for strategy in fallback_df['chunk_strategy'].unique():
        count = len(fallback_df[fallback_df['chunk_strategy'] == strategy])
        print(f"  • {strategy}: {count} chunks")
else:
    print("✓ No v1_hierarchical fallback errors found in this document.")

## Section 3: Understanding the Import Failure

**Error Message**: `v1_hierarchical→recursive (import_failed: NotebookImportException)`

This happens in `chunker.py` at this point:

```python
try:
    from common.fsr_v2.hierarchical_chunking_v1 import (
        hierarchical_semantic_chunking_from_snapshot,
        load_pdf_snapshot,
    )
except Exception as exc:
    self._actual_strategy = f"v1_hierarchical→recursive (import_failed: {type(exc).__name__})"
    return self._chunk_recursive(text)
```

**Root Cause Analysis**:

1. **NotebookImportException** = Databricks notebook import issue
2. **Why it happens**:
   - `hierarchical_chunking_v1` is likely a Databricks notebook, not a .py file
   - Python imports can't directly import notebook code
   - Need to use `dbutils.notebook.run()` or define as a module file

3. **Solutions**:
   - ✅ Convert hierarchical_chunking_v1 from notebook to Python module (.py)
   - ✅ Use dynamic notebook import via `dbutils`
   - ✅ Bundle it with the shared cluster libraries

In [ ]:
# Check: Where is hierarchical_chunking_v1 defined?
# This cell helps diagnose if the module exists and is accessible

import os
import subprocess

print("Searching for hierarchical_chunking_v1 in workspace...\n")

# Search in local workspace paths
search_paths = [
    "/home/u560060992/dbx/pw_sdg_ai_ser_repo",
    "/home/u560060992/dbx/pw_sdg_ai_ser_repo/common",
]

found_paths = []
for base_path in search_paths:
    if os.path.exists(base_path):
        # Search for .py files
        result = subprocess.run(
            ["find", base_path, "-name", "*hierarchical*", "-type", "f"],
            capture_output=True,
            text=True
        )
        if result.stdout:
            found_paths.extend(result.stdout.strip().split("\n"))

if found_paths:
    print("✓ Found hierarchical_chunking files:")
    for path in found_paths:
        if path:
            print(f"  • {path}")
            # Check if it's a Python module
            if path.endswith(".py"):
                print(f"    ✓ Is a Python module (.py)")
            elif path.endswith(".py.ipynb"):
                print(f"    ⚠ Is a Databricks notebook (.py.ipynb)")
            else:
                print(f"    ? Unknown type")
else:
    print("❌ NO hierarchical_chunking files found in workspace")
    print("\n   This explains the import failure!")
    print("   Action: Convert/create hierarchical_chunking_v1.py as a Python module")

## Section 4: Compare Chunking Strategies (All Available)

Compare metrics across strategies for the same document to understand if recursive-fallback is acceptable.

In [ ]:
# Compare all strategies for TARGET_DOC_ID
comparison_data = []

for strategy, table in CHUNK_TABLES.items():
    try:
        query = f"""
        SELECT 
            COUNT(*) as chunk_count,
            MIN(LENGTH(chunk_text)) as min_size,
            MAX(LENGTH(chunk_text)) as max_size,
            AVG(LENGTH(chunk_text)) as avg_size,
            COUNT(DISTINCT page_number) as unique_pages
        FROM {table}
        WHERE document_id = '{TARGET_DOC_ID}';
        """
        
        result = spark.sql(query).collect()[0]
        comparison_data.append({
            "Strategy": strategy,
            "Chunk Count": result[0],
            "Min Size": result[1],
            "Max Size": result[2],
            "Avg Size": int(result[3]) if result[3] else 0,
            "Pages": result[4],
        })
    except Exception as e:
        print(f"⚠ {strategy}: {str(e)[:80]}")

if comparison_data:
    comp_df = pd.DataFrame(comparison_data)
    print("\n📊 Strategy Comparison for", TARGET_DOC_ID)
    print("="*100)
    print(comp_df.to_string(index=False))
    print("\n✓ All strategies produced results for this document")
else:
    print("❌ No chunks found across any strategy table for this document")

## Section 5: Recommended Fix

The fallback logging is **working correctly**. The real issue is that `hierarchical_chunking_v1` cannot be imported.

### Option A: Quick Fix (Enable recursive fallback gracefully)
- Accept that v1_hierarchical falls back to recursive
- This is safe — recursive is a robust fallback strategy
- Chunks are still valid and retrievable

### Option B: Proper Fix (Make hierarchical_chunking_v1 importable)

**Step 1**: Locate hierarchical_chunking_v1
- Search for it in Databricks workspace notebooks
- Convert it to a Python file (common/fsr_v2/hierarchical_chunking_v1.py)
- Ensure it's on the cluster's Python path

**Step 2**: Update chunker.py import
- Fix the import statement if path is wrong
- Add better error messaging

**Step 3**: Re-run chunking job
- Set FSR_CHUNKING_STRATEGY=v1_hierarchical
- Reset chunk_status='pending' for test document
- Verify v1_hierarchical chunks appear without fallback reason

In [ ]:
# Final Diagnostic Summary
print("=" * 80)
print("FSR v2 IMPORT FAILURE DIAGNOSTIC SUMMARY")
print("=" * 80)

print("\n1️⃣  WHAT HAPPENED:")
print("   • Chunking strategy was declared as: v1_hierarchical")
print("   • But hierarchical_chunking_v1 could not be imported (NotebookImportException)")
print("   • Fallback: recursive strategy was used instead")
print("   • Result: chunk_strategy = 'v1_hierarchical→recursive (import_failed: NotebookImportException)'")

print("\n2️⃣  IS THIS A BUG?")
print("   ✓ NO - Fallback logging is working as designed")
print("   ✓ Chunks were still produced (recursive is safe fallback)")
print("   ✓ Data lineage is tracked (chunk_strategy shows what actually ran)")
print("   ✗ BUT - v1_hierarchical strategy was not available")

print("\n3️⃣  ROOT CAUSE:")
print("   ❓ hierarchical_chunking_v1 could be:")
print("      a) A Databricks notebook (not importable as Python module)")
print("      b) Missing from the Databricks workspace")
print("      c) Not on the cluster's Python path")
print("      d) In a different location than chunker.py expects")

print("\n4️⃣  ACTION ITEMS:")
print("   [ ] Find where hierarchical_chunking_v1 is defined")
print("   [ ] Determine if it's a notebook (.py.ipynb) or Python file (.py)")
print("   [ ] If notebook: Convert to Python module or use dbutils.notebook.run()")
print("   [ ] If Python file: Verify import path in chunker.py is correct")
print("   [ ] Add hierarchical_chunking_v1.py to common/fsr_v2/ if missing")
print("   [ ] Test by re-running chunking job with v1_hierarchical strategy")
print("   [ ] Verify chunk_strategy no longer shows 'import_failed'")

print("\n5️⃣  FOR NOW:")
print("   ✓ Chunks are valid (recursively chunked)")
print("   ✓ Continue testing with recursive, character, markdown, section strategies")
print("   ✓ Circle back to v1_hierarchical once module is available")

print("\n" + "=" * 80)